# Apartment Rental Offers in Germany — Reproducible Analysis

**Group G — Team 3 | Data analysis portfolio edition**

This notebook refactors the original team's Python analysis to correct broken notebook operations and make the workflow easier to run. It does not re-create or claim authorship of the team's Tableau visuals. The underlying data is not included in the repository.

## 1. Setup and dataset

The original project notebook used an ImmoScout24-based CSV available from a shared Google Drive file. Download access depends on the source owner's permissions. Save the dataset at `data/data_apartment.csv` relative to the repository root. The notebook supports Jupyter and Colab with an optional download step.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# Working from either the repository root or its notebooks/ subdirectory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "data_apartment.csv"
print("Expected dataset location:", DATA_PATH)

In [ ]:
# Optional: fetch the shared source CSV if it is not already present.
# Comment out this cell if you prefer downloading the CSV manually.
if not DATA_PATH.exists():
    import gdown
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    file_id = "1k3ZGqhQSavRmw5RkfCZc3dmQuYiyViV6"
    try:
        gdown.download(id=file_id, output=str(DATA_PATH), quiet=False)
    except Exception as exc:
        print("Automatic download unavailable:", exc)
        print("Download the CSV manually and place it at", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. See data/README.md for dataset instructions."
    )

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw data shape (rows, columns):", df.shape)
display(df.head())

## 2. Data inspection

Inspect field types and missing values before making cleaning decisions.

In [ ]:
df.info()
missing = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100)
    })
    .sort_values("missing_pct", ascending=False)
)
display(missing.head(20).round(2))

## 3. Data cleaning

Following the original notebook's approach, remove columns with more than 30% missing values, drop listings without `totalRent`, and standardize selected fields. `scoutId` is treated as a string identifier rather than a numeric measurement.

In [ ]:
df_clean = df.copy()
if "scoutId" in df_clean.columns:
    df_clean["scoutId"] = df_clean["scoutId"].astype("string")
if "date" in df_clean.columns:
    df_clean["date"] = pd.to_datetime(df_clean["date"], format="%b%y", errors="coerce")

missing_share = df_clean.isna().mean()
cols_to_drop = missing_share[missing_share > 0.30].index.tolist()
# Preserve the outcome variable regardless of its missingness; missing rows will be removed.
cols_to_drop = [c for c in cols_to_drop if c != "totalRent"]
df_clean = df_clean.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns with >30% missingness")

if "totalRent" not in df_clean.columns:
    raise KeyError("Expected column 'totalRent' not found in input CSV")
df_clean = df_clean.dropna(subset=["totalRent"]).copy()

if "condition" in df_clean.columns:
    df_clean["condition"] = df_clean["condition"].fillna("Other")

if "regio2" in df_clean.columns:
    top_20_regions = df_clean["regio2"].value_counts().head(20).index
    df_clean["regio2_grouped"] = df_clean["regio2"].where(
        df_clean["regio2"].isin(top_20_regions), "Other"
    )

if "scoutId" in df_clean.columns:
    duplicate_count = df_clean["scoutId"].duplicated().sum()
    print("Duplicate scoutId entries:", duplicate_count)
    # Preserve source rows; inspect duplicates before deciding whether to delete any.

print("Cleaned data shape:", df_clean.shape)

## 4. Exploratory data analysis

To keep charts responsive, use a reproducible sample of up to 10,000 listings. Summary statistics are descriptive, not causal.

In [ ]:
sample_n = min(10_000, len(df_clean))
df_sample = df_clean.sample(n=sample_n, random_state=42).copy()
print(f"Visualizing a sample of {len(df_sample):,} rows")
display(df_sample.describe(include="number").T.round(2))

In [ ]:
# Distribution of monthly total rent
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_sample["totalRent"].dropna(), bins=40)
ax.set(title="Total rent distribution", xlabel="Total rent (EUR)", ylabel="Listings")
plt.tight_layout()
plt.show()

In [ ]:
# Comparing regional rental prices
if "regio1" in df_sample.columns:
    top_regions = df_sample["regio1"].value_counts().head(12).index
    plot_data = df_sample[df_sample["regio1"].isin(top_regions)]
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=plot_data, x="regio1", y="totalRent", ax=ax, showfliers=False)
    ax.tick_params(axis="x", rotation=55)
    ax.set(title="Total rent across major regions", xlabel="Region", ylabel="Total rent (EUR)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Numeric correlation overview
features = [
    col for col in ["totalRent", "baseRent", "serviceCharge", "livingSpace", "noRooms", "yearConstructed"]
    if col in df_sample.columns and pd.api.types.is_numeric_dtype(df_sample[col])
]
if len(features) >= 2:
    corr = df_sample[features].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlation among selected numeric features")
    plt.tight_layout()
    plt.show()

In [ ]:
# Budget segments (thresholds adapted from the original notebook)
df_sample["budget_segment"] = pd.cut(
    df_sample["totalRent"],
    bins=[0, 1000, 2000, np.inf],
    labels=["low", "medium", "high"],
    include_lowest=True,
)
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_sample.dropna(subset=["budget_segment"]),
            x="budget_segment", y="totalRent", ax=ax, showfliers=False)
ax.set(title="Monthly rent by budget segment", xlabel="Budget segment", ylabel="Total rent (EUR)")
plt.tight_layout()
plt.show()

In [ ]:
# Building year, where available
if "yearConstructed" in df_sample.columns:
    building_year = pd.to_numeric(df_sample["yearConstructed"], errors="coerce").dropna()
    if not building_year.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.hist(building_year, bins=30)
        ax.set(title="Distribution of building construction year", xlabel="Year built", ylabel="Listings")
        plt.tight_layout()
        plt.show()

## 5. Interpretation and next steps

Use the plots to compare rental prices and property characteristics. The original team presentation included Tableau maps and regional rent comparisons, which are not regenerated by this notebook.

**Limitations:** The sample may not represent the entire housing market; missing-value filtering can affect representativeness; correlations do not demonstrate causality. The source CSV is external and may become unavailable.